> 📒 이 노트북은 [펭귄 데이터로 배우는 인공지능](https://yanghyeokjin33.github.io/Algorithm/) 수업에서 쓰는 실습 노트북이에요. 원본: [https://github.com/YANGHYEOKJIN33/Algorithm](https://github.com/YANGHYEOKJIN33/Algorithm)

# 🔍 결측치와 이상치 찾기

**목표**: 결측치가 있는지, 몇 개인지, 어디에 있는지 확인하고, 사분위수(IQR)와 상자그림으로 이상치를 찾는다.

연습 데이터에는 수업을 위해 일부러 숨겨 둔 함정이 세 가지 있어요. **빈칸**, **이상한 값**, **겹친 행**이에요. 하나씩 찾아봐요!

> **Colab 사용법** — 셀 왼쪽의 ▶ 단추를 누르거나 **Shift + Enter**를 누르면 그 셀이 실행돼요.
> 맨 위 셀부터 **차례로** 실행하세요. 코드를 고쳐 보고 싶다면 먼저 **파일 → 드라이브에 사본 저장**을 눌러 내 사본을 만들어 두세요.
> 파이썬을 몰라도 괜찮아요. 코드 위의 설명을 사이트의 의사코드와 한 줄씩 맞춰 읽으면 돼요.

- 수업용 펭귄 데이터를 인터넷 주소에서 바로 불러와요. 앞 시간에 크롤링한 파일과 같은 내용이에요.

In [ ]:
import pandas as pd
URL = 'https://raw.githubusercontent.com/YANGHYEOKJIN33/Algorithm/main/data/penguins.csv'
df = pd.read_csv(URL)          # 크롤링한 penguins.csv를 올렸다면 pd.read_csv('penguins.csv')
print(df.shape)
df.head()

## 1. 결측치 여부 — `isnull()`

칸이 비어 있으면 **True**, 값이 있으면 **False**로 표시한 표를 만들어요.

- 결과표는 원래 표와 크기가 같아요. True인 칸이 결측치예요. 인덱스 3, 6, 8~11 행을 살펴보세요.

In [ ]:
df.isnull().head(12)

## 2. 결측치 개수 — `isnull().sum()`

True는 1, False는 0으로 계산되기 때문에 **모두 더하면 개수**가 나와요.

- 열마다 빈칸이 몇 개인지 보여 줘요. 빈칸이 가장 많은 열은 성별이에요.

In [ ]:
print(df.isnull().sum())
print('전체 결측치:', df.isnull().sum().sum())

## 3. 결측치 위치 — 인덱스 찾기

`any(axis=1)`로 행마다 True가 하나라도 있는지 살피고, 그런 행의 인덱스를 모아요. 인덱스는 **0부터** 세는 자리 번호예요.

- df[조건]은 조건이 True인 행만 골라요.
- df.loc[인덱스 목록]은 그 행들을 보여 줘요.

In [ ]:
has_nan = df.isnull().any(axis=1)
where = df[has_nan].index
print('결측치가 있는 행의 인덱스:', list(where))
df.loc[where]

- df['몸무게'].isnull()은 몸무게 열만 검사해요.

In [ ]:
# 열 하나만 보고 싶을 때
df[df['몸무게'].isnull()]

## 4. 이상치 — 사분위수와 울타리

    Q1 = 25% 자리 값, Q3 = 75% 자리 값, IQR = Q3 − Q1
    아래 울타리 = Q1 − 1.5 × IQR,  위 울타리 = Q3 + 1.5 × IQR
    울타리 밖의 값 → 이상치

사이트 2-4에서는 아델리펭귄 13마리로 계산했지만, 여기서는 **345줄 전체**로 계산하기 때문에 Q1과 Q3 값이 달라요. 13마리로 하는 계산은 5번에서 다시 해 봐요.

- quantile(0.25)는 25% 자리(Q1)의 값이에요.
- | 는 "또는"이라는 뜻이에요. 아래 울타리보다 작거나, 위 울타리보다 큰 행을 골라요.
- 몸무게가 8200g인 아델리펭귄이 나와요. 3200을 8200으로 잘못 적은 값이에요!

In [ ]:
s = df['몸무게']
q1 = s.quantile(0.25)
q3 = s.quantile(0.75)
iqr = q3 - q1
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr
print(f'Q1={q1}, Q3={q3}, IQR={iqr}')
print(f'울타리: {low} ~ {high}')
df[(s < low) | (s > high)]          # 이상치인 행

## 5. 상자그림 ① — 사이트와 같은 13마리

상자그림에는 상자(Q1~Q3)와 가운데 선(중앙값), 수염이 있어요. 그리고 **수염 밖에 따로 찍힌 점이 이상치**예요.

먼저 사이트 2-4와 2-5에서 본 아델리펭귄 13마리만 골라 그려요. 상자 3450 ~ 3800, 가운데 선 3700, 수염 3250 ~ 4250, 따로 찍힌 점 8200으로 사이트 그림과 **값이 똑같아요**. 사이트는 가로로, Colab은 세로로 그린다는 점만 달라요.

- 그래프의 제목이나 축 이름에 쓴 한글이 깨지지 않도록 글꼴 도구를 설치해요. 처음 한 번은 1~2분쯤 걸릴 수 있어요.

In [ ]:
!pip install -q koreanize-matplotlib
try:
    import koreanize_matplotlib  # 그래프에 한글 글꼴을 입혀 줘요
except ImportError:
    print('한글 글꼴을 불러오지 못했어요. 그래프 글자가 네모로 보일 수 있어요.')
import matplotlib.pyplot as plt

- isin(목록)은 번호가 목록 안에 있는 행만 골라요.
- Q1=3450, Q2=3700, Q3=3800으로, 사이트 2-4에서 손으로 구한 값과 같아요.
- 맨 위에 홀로 떨어진 점이 8200g이에요. 사이트의 "실제 비율로" 그림을 세로로 세운 모양이에요.

In [ ]:
ids = [1, 2, 3, 5, 6, 7, 10, 11, 12, 13, 14, 16, 17]   # 사이트의 아델리펭귄 13마리 번호
small = df[df['번호'].isin(ids)]['몸무게']
print(small.quantile([0.25, 0.5, 0.75]))               # Q1, Q2, Q3

plt.figure(figsize=(4, 5))
plt.boxplot(small)
plt.title('아델리펭귄 13마리 몸무게')
plt.ylabel('몸무게 (g)')
plt.show()

## 6. 상자그림 ② — 345줄 전체 (값이 달라요)

이번에는 345줄 전체의 몸무게로 그려요. 빈칸 2개를 빼면 343개예요. **읽는 법은 똑같지만 값은 달라요.** 세 종이 섞여 있어서 값이 더 넓게 퍼져 있거든요.

| | 13마리 (사이트와 같음) | 345줄 전체 |
|---|---|---|
| 상자 Q1 ~ Q3 | 3450 ~ 3800 | 3550 ~ 4762.5 |
| 가운데 선 Q2 | 3700 | 4050 |
| 수염 | 3250 ~ 4250 | 2700 ~ 6300 |
| 따로 찍힌 점 | 8200 | 8200 |

- 빈칸은 그림에 나타낼 수 없어서 dropna()로 빼고 그려요.
- 상자 3550 ~ 4762.5, 가운데 선 4050, 수염 2700 ~ 6300으로, 위의 13마리 그림과 값이 달라요.
- 8200g은 전체로 봐도 위 울타리(6581.25)보다 커서 수염 밖에 점으로 찍혀요.

In [ ]:
plt.figure(figsize=(4, 5))
plt.boxplot(df['몸무게'].dropna())
plt.title('345줄 전체 몸무게')
plt.ylabel('몸무게 (g)')
plt.show()

- by='종'을 주면 종마다 상자를 따로 그려요. 젠투가 가장 무겁다는 점도 한눈에 보여요.

In [ ]:
# 종마다 나눠 그리면 차이가 더 잘 보여요
df.boxplot(column='몸무게', by='종', figsize=(6, 4))
plt.suptitle('')
plt.title('종별 몸무게')
plt.show()

## 🧩 확인해 봐요 · 한 번에 보기

- `df.describe()`를 실행하면 개수(count), 평균, 사분위수, 최솟값, 최댓값이 한 번에 나와요. count가 345보다 작은 열은 왜 그럴까요?
- `df.duplicated().sum()`을 실행해 보세요. 겹친 행이 몇 개인가요? 겹친 행은 다음 시간 🧹 전처리에서 지워요.

- describe()는 숫자 열을 한눈에 요약한 표를 보여 줘요.

In [ ]:
print('겹친 행:', df.duplicated().sum())
df.describe()